In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql import Row
from datetime import date, datetime
from decimal import Decimal
import random

In [2]:
spark = SparkSession.builder.appName("iceberg_integration").getOrCreate()
print(f"Spark version: {spark.version}")
print(f"Default catalog: {spark.conf.get('spark.sql.defaultCatalog')}")

Spark version: 3.5.0
Default catalog: dev_catalog


In [3]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS sales")
spark.sql("SHOW NAMESPACES").show()

+---------+
|namespace|
+---------+
|    sales|
+---------+



In [4]:
# Define schema
schema = StructType([
    StructField("transaction_id", StringType(), False),
    StructField("customer_id", StringType(), False),
    StructField("product_id", StringType(), True),
    StructField("amount", DecimalType(10, 2), False),
    StructField("quantity", IntegerType(), False),
    StructField("transaction_date", DateType(), False),
    StructField("region", StringType(), False)
])

# Create empty DataFrame
empty_df = spark.createDataFrame([], schema)

# Drop first to reset
spark.sql("DROP TABLE IF EXISTS sales.transactions PURGE")

# Write as Iceberg table
empty_df.writeTo("sales.transactions") \
    .partitionedBy(
        col("region"),
        days(col('transaction_date'))
    ) \
    .tableProperty("format-version", "2") \
    .createOrReplace()

spark.sql("DESCRIBE EXTENDED sales.transactions").show(200, truncate=False)

+----------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+
|col_name                    |data_type                                                                                                                                                       |comment|
+----------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+
|transaction_id              |string                                                                                                                                                          |NULL   |
|customer_id                 |string                                                                                                                                                          |NULL   |


In [5]:
data = [
    Row("TXN001", "CUST001", "PROD001", Decimal("99.99"), 2, date(2024, 1, 15), "north"),
    Row("TXN002", "CUST002", "PROD002", Decimal("149.50"), 1, date(2024, 1, 15), "south"),
    Row("TXN003", "CUST003", "PROD003", Decimal("200.00"), 3, date(2024, 1, 16), "east"),
    Row("TXN004", "CUST004", "PROD004", Decimal("120.25"), 1, date(2024, 1, 16), "west"),
    Row("TXN005", "CUST005", "PROD005", Decimal("75.75"), 5, date(2024, 1, 17), "north"),
    Row("TXN006", "CUST006", "PROD006", Decimal("300.00"), 2, date(2024, 1, 17), "south"),
    Row("TXN007", "CUST007", "PROD007", Decimal("50.50"), 4, date(2024, 1, 18), "east"),
    Row("TXN008", "CUST008", "PROD008", Decimal("180.90"), 1, date(2024, 1, 18), "west"),
    Row("TXN009", "CUST009", "PROD009", Decimal("220.20"), 2, date(2024, 1, 19), "north"),
    Row("TXN010", "CUST010", "PROD010", Decimal("99.99"), 3, date(2024, 1, 19), "south"),
]

df = spark.createDataFrame(data, schema)
df.writeTo("sales.transactions").append()

spark.sql("SELECT * FROM sales.transactions").show()

+--------------+-----------+----------+------+--------+----------------+------+
|transaction_id|customer_id|product_id|amount|quantity|transaction_date|region|
+--------------+-----------+----------+------+--------+----------------+------+
|        TXN007|    CUST007|   PROD007| 50.50|       4|      2024-01-18|  east|
|        TXN008|    CUST008|   PROD008|180.90|       1|      2024-01-18|  west|
|        TXN004|    CUST004|   PROD004|120.25|       1|      2024-01-16|  west|
|        TXN003|    CUST003|   PROD003|200.00|       3|      2024-01-16|  east|
|        TXN009|    CUST009|   PROD009|220.20|       2|      2024-01-19| north|
|        TXN010|    CUST010|   PROD010| 99.99|       3|      2024-01-19| south|
|        TXN001|    CUST001|   PROD001| 99.99|       2|      2024-01-15| north|
|        TXN006|    CUST006|   PROD006|300.00|       2|      2024-01-17| south|
|        TXN005|    CUST005|   PROD005| 75.75|       5|      2024-01-17| north|
|        TXN002|    CUST002|   PROD002|1

In [6]:
data2 = [
    Row("TXN011", "CUST011", "PROD011", Decimal("125.50"), 2, date(2024, 1, 20), "north"),
    Row("TXN012", "CUST012", "PROD012", Decimal("130.00"), 1, date(2024, 1, 19), "north"), 
]

df2 = spark.createDataFrame(data2, schema)

df2.writeTo("sales.transactions") \
    .option("overwrite-mode", "dynamic") \
    .overwritePartitions()

spark.sql("SELECT * FROM sales.transactions ORDER BY region, transaction_date").show()

+--------------+-----------+----------+------+--------+----------------+------+
|transaction_id|customer_id|product_id|amount|quantity|transaction_date|region|
+--------------+-----------+----------+------+--------+----------------+------+
|        TXN003|    CUST003|   PROD003|200.00|       3|      2024-01-16|  east|
|        TXN007|    CUST007|   PROD007| 50.50|       4|      2024-01-18|  east|
|        TXN001|    CUST001|   PROD001| 99.99|       2|      2024-01-15| north|
|        TXN005|    CUST005|   PROD005| 75.75|       5|      2024-01-17| north|
|        TXN012|    CUST012|   PROD012|130.00|       1|      2024-01-19| north|
|        TXN011|    CUST011|   PROD011|125.50|       2|      2024-01-20| north|
|        TXN002|    CUST002|   PROD002|149.50|       1|      2024-01-15| south|
|        TXN006|    CUST006|   PROD006|300.00|       2|      2024-01-17| south|
|        TXN010|    CUST010|   PROD010| 99.99|       3|      2024-01-19| south|
|        TXN004|    CUST004|   PROD004|1

In [7]:
spark.table("sales.transactions").show()

+--------------+-----------+----------+------+--------+----------------+------+
|transaction_id|customer_id|product_id|amount|quantity|transaction_date|region|
+--------------+-----------+----------+------+--------+----------------+------+
|        TXN007|    CUST007|   PROD007| 50.50|       4|      2024-01-18|  east|
|        TXN012|    CUST012|   PROD012|130.00|       1|      2024-01-19| north|
|        TXN008|    CUST008|   PROD008|180.90|       1|      2024-01-18|  west|
|        TXN011|    CUST011|   PROD011|125.50|       2|      2024-01-20| north|
|        TXN004|    CUST004|   PROD004|120.25|       1|      2024-01-16|  west|
|        TXN003|    CUST003|   PROD003|200.00|       3|      2024-01-16|  east|
|        TXN010|    CUST010|   PROD010| 99.99|       3|      2024-01-19| south|
|        TXN001|    CUST001|   PROD001| 99.99|       2|      2024-01-15| north|
|        TXN006|    CUST006|   PROD006|300.00|       2|      2024-01-17| south|
|        TXN005|    CUST005|   PROD005| 

In [8]:
spark.sql("""
    SELECT * 
    FROM my_catalog.sales.transactions
    WHERE region = 'north'
      AND transaction_date >= '2024-01-01'
""").show()

+--------------+-----------+----------+------+--------+----------------+------+
|transaction_id|customer_id|product_id|amount|quantity|transaction_date|region|
+--------------+-----------+----------+------+--------+----------------+------+
|        TXN012|    CUST012|   PROD012|130.00|       1|      2024-01-19| north|
|        TXN011|    CUST011|   PROD011|125.50|       2|      2024-01-20| north|
|        TXN001|    CUST001|   PROD001| 99.99|       2|      2024-01-15| north|
|        TXN005|    CUST005|   PROD005| 75.75|       5|      2024-01-17| north|
+--------------+-----------+----------+------+--------+----------------+------+



In [9]:
spark.sql("""
    SELECT COUNT(*) 
    FROM my_catalog.sales.transactions
""").collect()[0][0]

11

In [10]:
df_snapshot = spark.sql("SELECT * FROM sales.transactions.snapshots")
df_snapshot.show()

+--------------------+-------------------+-------------------+---------+--------------------+--------------------+
|        committed_at|        snapshot_id|          parent_id|operation|       manifest_list|             summary|
+--------------------+-------------------+-------------------+---------+--------------------+--------------------+
|2026-04-07 09:45:...|6037324567914049945|               NULL|   append|s3://lakehouse/sa...|{spark.app.id -> ...|
|2026-04-07 09:45:...|1513603331442321162|6037324567914049945|   append|s3://lakehouse/sa...|{spark.app.id -> ...|
|2026-04-07 09:45:...|5282394550707250568|1513603331442321162|overwrite|s3://lakehouse/sa...|{spark.app.id -> ...|
+--------------------+-------------------+-------------------+---------+--------------------+--------------------+



In [11]:
for value in df_snapshot.select("committed_at").collect():
    print(value[0])

2026-04-07 09:45:37.780000
2026-04-07 09:45:45.945000
2026-04-07 09:45:50.013000


In [12]:
spark.read \
    .option("snapshot-id", 6037324567914049945) \
    .table("my_catalog.sales.transactions") \
    .show()

+--------------+-----------+----------+------+--------+----------------+------+
|transaction_id|customer_id|product_id|amount|quantity|transaction_date|region|
+--------------+-----------+----------+------+--------+----------------+------+
+--------------+-----------+----------+------+--------+----------------+------+



In [13]:
dt = datetime(2026, 4, 7, 9, 45, 40)
ts_ms = int(dt.timestamp() * 1000)  # ← phải là milliseconds, ví dụ: 1776254025000

spark.read \
    .option("as-of-timestamp", ts_ms) \
    .table("my_catalog.sales.transactions") \
    .show()

+--------------+-----------+----------+------+--------+----------------+------+
|transaction_id|customer_id|product_id|amount|quantity|transaction_date|region|
+--------------+-----------+----------+------+--------+----------------+------+
+--------------+-----------+----------+------+--------+----------------+------+



In [14]:
spark.sql("SELECT * FROM sales.transactions WHERE region = 'north' AND transaction_date < '2024-01-19'").show()

spark.sql("""
    UPDATE my_catalog.sales.transactions
    SET amount = amount * 1.1
    WHERE region = 'north' AND transaction_date < '2024-01-19'
""")

spark.sql("SELECT * FROM sales.transactions WHERE region = 'north' AND transaction_date < '2024-01-19'").show()

+--------------+-----------+----------+------+--------+----------------+------+
|transaction_id|customer_id|product_id|amount|quantity|transaction_date|region|
+--------------+-----------+----------+------+--------+----------------+------+
|        TXN001|    CUST001|   PROD001| 99.99|       2|      2024-01-15| north|
|        TXN005|    CUST005|   PROD005| 75.75|       5|      2024-01-17| north|
+--------------+-----------+----------+------+--------+----------------+------+

+--------------+-----------+----------+------+--------+----------------+------+
|transaction_id|customer_id|product_id|amount|quantity|transaction_date|region|
+--------------+-----------+----------+------+--------+----------------+------+
|        TXN001|    CUST001|   PROD001|109.99|       2|      2024-01-15| north|
|        TXN005|    CUST005|   PROD005| 83.33|       5|      2024-01-17| north|
+--------------+-----------+----------+------+--------+----------------+------+



In [15]:
spark.sql("""
    DELETE FROM my_catalog.sales.transactions
    WHERE transaction_id = 'TXN001'
""")

spark.sql("SELECT * FROM sales.transactions").show()

+--------------+-----------+----------+------+--------+----------------+------+
|transaction_id|customer_id|product_id|amount|quantity|transaction_date|region|
+--------------+-----------+----------+------+--------+----------------+------+
|        TXN007|    CUST007|   PROD007| 50.50|       4|      2024-01-18|  east|
|        TXN008|    CUST008|   PROD008|180.90|       1|      2024-01-18|  west|
|        TXN012|    CUST012|   PROD012|130.00|       1|      2024-01-19| north|
|        TXN004|    CUST004|   PROD004|120.25|       1|      2024-01-16|  west|
|        TXN005|    CUST005|   PROD005| 83.33|       5|      2024-01-17| north|
|        TXN003|    CUST003|   PROD003|200.00|       3|      2024-01-16|  east|
|        TXN011|    CUST011|   PROD011|125.50|       2|      2024-01-20| north|
|        TXN010|    CUST010|   PROD010| 99.99|       3|      2024-01-19| south|
|        TXN006|    CUST006|   PROD006|300.00|       2|      2024-01-17| south|
|        TXN002|    CUST002|   PROD002|1

In [16]:
spark.sql("""
    DELETE FROM my_catalog.sales.transactions
    WHERE amount < 90
""")

spark.sql("SELECT * FROM sales.transactions").show()

+--------------+-----------+----------+------+--------+----------------+------+
|transaction_id|customer_id|product_id|amount|quantity|transaction_date|region|
+--------------+-----------+----------+------+--------+----------------+------+
|        TXN012|    CUST012|   PROD012|130.00|       1|      2024-01-19| north|
|        TXN008|    CUST008|   PROD008|180.90|       1|      2024-01-18|  west|
|        TXN011|    CUST011|   PROD011|125.50|       2|      2024-01-20| north|
|        TXN004|    CUST004|   PROD004|120.25|       1|      2024-01-16|  west|
|        TXN003|    CUST003|   PROD003|200.00|       3|      2024-01-16|  east|
|        TXN010|    CUST010|   PROD010| 99.99|       3|      2024-01-19| south|
|        TXN006|    CUST006|   PROD006|300.00|       2|      2024-01-17| south|
|        TXN002|    CUST002|   PROD002|149.50|       1|      2024-01-15| south|
+--------------+-----------+----------+------+--------+----------------+------+



In [17]:
# Define schema
schema_with_op = StructType([
    StructField("transaction_id", StringType(), False),
    StructField("customer_id", StringType(), False),
    StructField("product_id", StringType(), True),
    StructField("amount", DecimalType(10, 2), False),
    StructField("quantity", IntegerType(), False),
    StructField("transaction_date", DateType(), False),
    StructField("region", StringType(), False),
    StructField("op", StringType(), False)
])

# Kafka / Debezium emit events: op = I / U / D
cdc_df = spark.createDataFrame([
    Row("TXN008", "CUST001", "PROD001", Decimal("110.00"), 1, date(2024,1,15), "north", "U"),  # update
    Row("TXN002", "CUST002", "PROD002", Decimal("200.00"), 2, date(2024,1,16), "south", "D"),  # delete
    Row("TXN099", "CUST099", "PROD099", Decimal("999.00"), 5, date(2024,2,1),  "east",  "I"),  # insert
], schema_with_op)

cdc_df.createOrReplaceTempView("cdc_source")

spark.sql("""
    MERGE INTO my_catalog.sales.transactions t
    USING (
        SELECT transaction_id, customer_id, product_id,
               amount, quantity, transaction_date, region, op
        FROM cdc_source
    ) s
    ON t.transaction_id = s.transaction_id

    WHEN MATCHED AND s.op = 'D' THEN DELETE
    WHEN MATCHED AND s.op = 'U' THEN
        UPDATE SET t.amount = s.amount, t.quantity = s.quantity
    WHEN NOT MATCHED AND s.op = 'I' THEN
        INSERT (transaction_id, customer_id, product_id,
                amount, quantity, transaction_date, region)
        VALUES (s.transaction_id, s.customer_id, s.product_id,
                s.amount, s.quantity, s.transaction_date, s.region)
""")

spark.sql("SELECT * FROM sales.transactions").show()

+--------------+-----------+----------+------+--------+----------------+------+
|transaction_id|customer_id|product_id|amount|quantity|transaction_date|region|
+--------------+-----------+----------+------+--------+----------------+------+
|        TXN012|    CUST012|   PROD012|130.00|       1|      2024-01-19| north|
|        TXN004|    CUST004|   PROD004|120.25|       1|      2024-01-16|  west|
|        TXN003|    CUST003|   PROD003|200.00|       3|      2024-01-16|  east|
|        TXN011|    CUST011|   PROD011|125.50|       2|      2024-01-20| north|
|        TXN010|    CUST010|   PROD010| 99.99|       3|      2024-01-19| south|
|        TXN006|    CUST006|   PROD006|300.00|       2|      2024-01-17| south|
|        TXN008|    CUST008|   PROD008|110.00|       1|      2024-01-18|  west|
|        TXN099|    CUST099|   PROD099|999.00|       5|      2024-02-01|  east|
+--------------+-----------+----------+------+--------+----------------+------+



In [18]:
df_snapshot = spark.sql("SELECT * FROM sales.transactions.snapshots")
df_snapshot.show()

+--------------------+-------------------+-------------------+---------+--------------------+--------------------+
|        committed_at|        snapshot_id|          parent_id|operation|       manifest_list|             summary|
+--------------------+-------------------+-------------------+---------+--------------------+--------------------+
|2026-04-07 09:45:...|6037324567914049945|               NULL|   append|s3://lakehouse/sa...|{spark.app.id -> ...|
|2026-04-07 09:45:...|1513603331442321162|6037324567914049945|   append|s3://lakehouse/sa...|{spark.app.id -> ...|
|2026-04-07 09:45:...|5282394550707250568|1513603331442321162|overwrite|s3://lakehouse/sa...|{spark.app.id -> ...|
|2026-04-07 09:46:...|1057291516723888781|5282394550707250568|overwrite|s3://lakehouse/sa...|{spark.app.id -> ...|
|2026-04-07 09:46:...|8942237634462046839|1057291516723888781|   delete|s3://lakehouse/sa...|{spark.app.id -> ...|
|2026-04-07 09:46:...|7192190069353111341|8942237634462046839|   delete|s3://lak

In [19]:
spark.sql(f"""
    CALL my_catalog.system.rollback_to_snapshot(
        'sales.transactions', 1513603331442321162
    )
""")

spark.sql("SELECT * FROM sales.transactions").show()

+--------------+-----------+----------+------+--------+----------------+------+
|transaction_id|customer_id|product_id|amount|quantity|transaction_date|region|
+--------------+-----------+----------+------+--------+----------------+------+
|        TXN007|    CUST007|   PROD007| 50.50|       4|      2024-01-18|  east|
|        TXN008|    CUST008|   PROD008|180.90|       1|      2024-01-18|  west|
|        TXN004|    CUST004|   PROD004|120.25|       1|      2024-01-16|  west|
|        TXN003|    CUST003|   PROD003|200.00|       3|      2024-01-16|  east|
|        TXN009|    CUST009|   PROD009|220.20|       2|      2024-01-19| north|
|        TXN010|    CUST010|   PROD010| 99.99|       3|      2024-01-19| south|
|        TXN001|    CUST001|   PROD001| 99.99|       2|      2024-01-15| north|
|        TXN006|    CUST006|   PROD006|300.00|       2|      2024-01-17| south|
|        TXN005|    CUST005|   PROD005| 75.75|       5|      2024-01-17| north|
|        TXN002|    CUST002|   PROD002|1

In [20]:
spark.sql("DROP TABLE IF EXISTS my_catalog.sales.orders PURGE")

spark.sql("""
CREATE TABLE my_catalog.sales.orders (
    order_id STRING,
    order_date DATE,
    region STRING
) USING iceberg
PARTITIONED BY (region)
""")

DataFrame[]

In [21]:
regions = ["North", "South", "East", "West"]
dates_old = ["2026-04-01", "2026-04-02", "2026-04-03"]

data_old = [(f"o{i}_{r}_{d}", d, r) 
            for i in range(1, 101)  # 100 record mỗi region/date
            for r in regions
            for d in dates_old]

df_old = spark.createDataFrame(data_old, ["order_id", "order_date", "region"])
df_old = df_old.withColumn("order_date", to_date(col("order_date"), "yyyy-MM-dd"))
df_old.writeTo("my_catalog.sales.orders").append()

spark.sql("SELECT * FROM my_catalog.sales.orders LIMIT 20").show(truncate=False)

+------------------+----------+------+
|order_id          |order_date|region|
+------------------+----------+------+
|o1_West_2026-04-01|2026-04-01|West  |
|o1_West_2026-04-02|2026-04-02|West  |
|o1_West_2026-04-03|2026-04-03|West  |
|o2_West_2026-04-01|2026-04-01|West  |
|o2_West_2026-04-02|2026-04-02|West  |
|o2_West_2026-04-03|2026-04-03|West  |
|o3_West_2026-04-01|2026-04-01|West  |
|o3_West_2026-04-02|2026-04-02|West  |
|o3_West_2026-04-03|2026-04-03|West  |
|o4_West_2026-04-01|2026-04-01|West  |
|o4_West_2026-04-02|2026-04-02|West  |
|o4_West_2026-04-03|2026-04-03|West  |
|o5_West_2026-04-01|2026-04-01|West  |
|o5_West_2026-04-02|2026-04-02|West  |
|o5_West_2026-04-03|2026-04-03|West  |
|o6_West_2026-04-01|2026-04-01|West  |
|o6_West_2026-04-02|2026-04-02|West  |
|o6_West_2026-04-03|2026-04-03|West  |
|o7_West_2026-04-01|2026-04-01|West  |
|o7_West_2026-04-02|2026-04-02|West  |
+------------------+----------+------+



In [22]:
spark.sql("""
ALTER TABLE my_catalog.sales.orders 
ADD PARTITION FIELD day(order_date)
""")

DataFrame[]

In [23]:
dates_new = ["2026-04-02", "2026-04-03", "2026-04-04"]
data_new = [(f"n{i}_{r}_{d}", d, r) 
            for i in range(101, 201)  # 100 record mỗi region/date
            for r in regions
            for d in dates_new]

df_new = spark.createDataFrame(data_new, ["order_id", "order_date", "region"])
df_new = df_new.withColumn("order_date", to_date(col("order_date"), "yyyy-MM-dd"))
df_new.writeTo("my_catalog.sales.orders").append()

spark.sql("""
SELECT *, day(order_date) AS day_partition 
FROM my_catalog.sales.orders 
LIMIT 20
""").show(truncate=False)

+------------------+----------+------+-------------+
|order_id          |order_date|region|day_partition|
+------------------+----------+------+-------------+
|o1_West_2026-04-01|2026-04-01|West  |1            |
|o1_West_2026-04-02|2026-04-02|West  |2            |
|o1_West_2026-04-03|2026-04-03|West  |3            |
|o2_West_2026-04-01|2026-04-01|West  |1            |
|o2_West_2026-04-02|2026-04-02|West  |2            |
|o2_West_2026-04-03|2026-04-03|West  |3            |
|o3_West_2026-04-01|2026-04-01|West  |1            |
|o3_West_2026-04-02|2026-04-02|West  |2            |
|o3_West_2026-04-03|2026-04-03|West  |3            |
|o4_West_2026-04-01|2026-04-01|West  |1            |
|o4_West_2026-04-02|2026-04-02|West  |2            |
|o4_West_2026-04-03|2026-04-03|West  |3            |
|o5_West_2026-04-01|2026-04-01|West  |1            |
|o5_West_2026-04-02|2026-04-02|West  |2            |
|o5_West_2026-04-03|2026-04-03|West  |3            |
|o6_West_2026-04-01|2026-04-01|West  |1       

In [24]:
spark.sql("""
ALTER TABLE my_catalog.sales.orders
REPLACE PARTITION FIELD region WITH bucket(10, region)
""")

DataFrame[]

In [25]:
regions = ["North", "South", "East", "West"]
dates = ["2026-04-01", "2026-04-02", "2026-04-03", "2026-04-04"]

data_new2 = [
    (f"new{i}_{random.choice(regions)}", random.choice(dates), random.choice(regions))
    for i in range(1, 1001)
]

data_new2 = spark.createDataFrame(data_new2, ["order_id", "order_date", "region"])
data_new2 = data_new2.withColumn("order_date", to_date(col("order_date"), "yyyy-MM-dd"))

data_new2.writeTo("my_catalog.sales.orders").append()

spark.sql("""
SELECT order_id, order_date, region, bucket(10, region) AS bucket_partition
FROM my_catalog.sales.orders
ORDER BY order_id
LIMIT 20
""").show(truncate=False)

+---------------------+----------+------+----------------+
|order_id             |order_date|region|bucket_partition|
+---------------------+----------+------+----------------+
|n101_East_2026-04-02 |2026-04-02|East  |0               |
|n101_East_2026-04-03 |2026-04-03|East  |0               |
|n101_East_2026-04-04 |2026-04-04|East  |0               |
|n101_North_2026-04-02|2026-04-02|North |6               |
|n101_North_2026-04-03|2026-04-03|North |6               |
|n101_North_2026-04-04|2026-04-04|North |6               |
|n101_South_2026-04-02|2026-04-02|South |2               |
|n101_South_2026-04-03|2026-04-03|South |2               |
|n101_South_2026-04-04|2026-04-04|South |2               |
|n101_West_2026-04-02 |2026-04-02|West  |3               |
|n101_West_2026-04-03 |2026-04-03|West  |3               |
|n101_West_2026-04-04 |2026-04-04|West  |3               |
|n102_East_2026-04-02 |2026-04-02|East  |0               |
|n102_East_2026-04-03 |2026-04-03|East  |0              

In [26]:
spark.sql("""
ALTER TABLE my_catalog.sales.orders
DROP PARTITION FIELD day(order_date)
""")

DataFrame[]

In [27]:
regions = ["North", "South", "East", "West"]
dates = ["2026-04-01","2026-04-02","2026-04-03","2026-04-04"]

data_new3 = [
    (f"new{i}_{random.choice(regions)}", random.choice(dates), random.choice(regions))
    for i in range(1101, 1201)
]

data_new3 = spark.createDataFrame(data_new3, ["order_id", "order_date", "region"])
data_new3 = data_new3.withColumn("order_date", to_date(col("order_date"), "yyyy-MM-dd"))

# Ghi dữ liệu mới vào bảng
data_new3.writeTo("my_catalog.sales.orders").append()

# Kiểm tra bucket partition
spark.sql("""
SELECT order_id, order_date, region, bucket(10, region) AS bucket_region
FROM my_catalog.sales.orders
ORDER BY order_id
LIMIT 20
""").show(truncate=False)

+---------------------+----------+------+-------------+
|order_id             |order_date|region|bucket_region|
+---------------------+----------+------+-------------+
|n101_East_2026-04-02 |2026-04-02|East  |0            |
|n101_East_2026-04-03 |2026-04-03|East  |0            |
|n101_East_2026-04-04 |2026-04-04|East  |0            |
|n101_North_2026-04-02|2026-04-02|North |6            |
|n101_North_2026-04-03|2026-04-03|North |6            |
|n101_North_2026-04-04|2026-04-04|North |6            |
|n101_South_2026-04-02|2026-04-02|South |2            |
|n101_South_2026-04-03|2026-04-03|South |2            |
|n101_South_2026-04-04|2026-04-04|South |2            |
|n101_West_2026-04-02 |2026-04-02|West  |3            |
|n101_West_2026-04-03 |2026-04-03|West  |3            |
|n101_West_2026-04-04 |2026-04-04|West  |3            |
|n102_East_2026-04-02 |2026-04-02|East  |0            |
|n102_East_2026-04-03 |2026-04-03|East  |0            |
|n102_East_2026-04-04 |2026-04-04|East  |0      

In [28]:
data = [
    (i, f"product_{random.randint(1,100)}", random.randint(1,10)*100, f"2026-04-{random.randint(1,7):02d}")
    for i in range(50)
]
columns = ["transaction_id", "product_id", "amount", "date"]
df = spark.createDataFrame(data, schema=columns)

# Lưu vào Iceberg table
spark.sql("DROP TABLE IF EXISTS my_catalog.sales.trans PURGE")

df.writeTo("my_catalog.sales.trans").tableProperty("format-version", "2").createOrReplace()

In [29]:
spark.sql("""
CALL my_catalog.system.rewrite_data_files(
    table => 'sales.trans',
    options => map(
        'target-file-size-bytes', '134217728',  -- 128 MB
        'min-input-files', '2'
    )
)
""")

DataFrame[rewritten_data_files_count: int, added_data_files_count: int, rewritten_bytes_count: bigint, failed_data_files_count: int]